# Performance Analytics
## Mutual Fund Analytics Project

### Objective
Analyze mutual fund performance using return, risk, benchmark and ranking metrics.


## Task 1: Daily Returns

Calculate daily percentage returns using pct_change().


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



file_path = r"D:\Mutual_Fund_Analytics\data\processed\02_nav_history_clean.csv"

df = pd.read_csv(file_path)

# ---------------------------------
# Convert Date Column to DateTime
# ---------------------------------

df["date"] = pd.to_datetime(df["date"])

# ---------------------------------
# Sort Data by AMFI Code and Date
# ---------------------------------

df = df.sort_values(
    by=["amfi_code", "date"]
)

# ---------------------------------
# Compute Daily Returns
# ---------------------------------

df["daily_return"] = (
    df.groupby("amfi_code")["nav"]
      .pct_change()
)

# ---------------------------------
# Remove First Record of Each Fund
# ---------------------------------

df = df.dropna(subset=["daily_return"])

# ---------------------------------
# Display First 10 Records
# ---------------------------------

print("First 10 Daily Returns:\n")

print(
    df[
        ["amfi_code", "date", "nav", "daily_return"]
    ].head(10)
)

# ---------------------------------
# Summary Statistics
# ---------------------------------

print("\nSummary Statistics:\n")

print(df["daily_return"].describe())

# ---------------------------------
# Plot Daily Return Distribution
# ---------------------------------

plt.figure(figsize=(10,6))

sns.histplot(
    df["daily_return"],
    bins=50,
    kde=True
)

plt.title(
    "Distribution of Daily Returns",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Daily Return")

plt.ylabel("Frequency")

plt.tight_layout()

# ---------------------------------
# Save Daily Returns Dataset
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\daily_returns.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nDaily Returns file saved successfully!")

print(output_file)

# ---------------------------------
# Display Histogram
# ---------------------------------

plt.show()


## Task 2: CAGR

Compute 1-year, 3-year and 5-year CAGR.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# ---------------------------------
# Load Cleaned NAV Dataset
# ---------------------------------

file_path = r"D:\Mutual_Fund_Analytics\data\processed\02_nav_history_clean.csv"

df = pd.read_csv(file_path)

# ---------------------------------
# Convert Date Column
# ---------------------------------

df["date"] = pd.to_datetime(df["date"])

# ---------------------------------
# Sort Data
# ---------------------------------

df = df.sort_values(
    by=["amfi_code", "date"]
)

# ---------------------------------
# CAGR Function
# ---------------------------------

def calculate_cagr(start_nav, end_nav, years):

    if start_nav <= 0 or end_nav <= 0:
        return np.nan

    return ((end_nav / start_nav) ** (1 / years)) - 1

# ---------------------------------
# Calculate CAGR for Each Fund
# ---------------------------------

results = []

for fund in df["amfi_code"].unique():

    fund_data = df[df["amfi_code"] == fund]

    fund_data = fund_data.sort_values("date")

    if len(fund_data) < 2:
        continue

    latest_nav = fund_data.iloc[-1]["nav"]

    cagr_1 = np.nan
    cagr_3 = np.nan
    cagr_5 = np.nan

    if len(fund_data) >= 252:
        start_nav = fund_data.iloc[-252]["nav"]
        cagr_1 = calculate_cagr(start_nav, latest_nav, 1)

    if len(fund_data) >= 756:
        start_nav = fund_data.iloc[-756]["nav"]
        cagr_3 = calculate_cagr(start_nav, latest_nav, 3)

    if len(fund_data) >= 1260:
        start_nav = fund_data.iloc[-1260]["nav"]
        cagr_5 = calculate_cagr(start_nav, latest_nav, 5)

    results.append({

        "amfi_code": fund,

        "cagr_1yr": cagr_1,

        "cagr_3yr": cagr_3,

        "cagr_5yr": cagr_5

    })

# ---------------------------------
# Create Comparison Table
# ---------------------------------

cagr_df = pd.DataFrame(results)

# Convert to Percentage

cagr_df["cagr_1yr"] = cagr_df["cagr_1yr"] * 100
cagr_df["cagr_3yr"] = cagr_df["cagr_3yr"] * 100
cagr_df["cagr_5yr"] = cagr_df["cagr_5yr"] * 100

# ---------------------------------
# Display Results
# ---------------------------------

print("\nCAGR Comparison Table\n")

print(cagr_df)

# ---------------------------------
# Save CSV
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\cagr_comparison.csv"

cagr_df.to_csv(
    output_file,
    index=False
)

print("\nCAGR Comparison Table saved successfully!")

print(output_file)
plt.figure(figsize=(12,6))

cagr_df.set_index("amfi_code")[[
    "cagr_1yr",
    "cagr_3yr",
    "cagr_5yr"
]].plot(
    kind="bar",
    figsize=(14,6)
)

plt.title(
    "CAGR Comparison of All Mutual Funds",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("AMFI Code")

plt.ylabel("CAGR (%)")

plt.xticks(rotation=90)

plt.tight_layout()

plt.show()


## Task 3: Sharpe Ratio

Compute risk-adjusted return using a 6.5% risk-free rate.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# ---------------------------------
# Load Daily Returns Dataset
# ---------------------------------

file_path = r"D:\Mutual_Fund_Analytics\data\processed\daily_returns.csv"

df = pd.read_csv(file_path)

# ---------------------------------
# Risk-Free Rate
# ---------------------------------

risk_free_rate = 0.065

daily_rf = risk_free_rate / 252

# ---------------------------------
# Calculate Sharpe Ratio
# ---------------------------------

results = []

for fund in df["amfi_code"].unique():

    fund_data = df[df["amfi_code"] == fund]

    mean_return = fund_data["daily_return"].mean()

    std_return = fund_data["daily_return"].std()

    if std_return == 0:
        sharpe = np.nan
    else:
        sharpe = (
            (mean_return - daily_rf)
            / std_return
        ) * np.sqrt(252)

    results.append({

        "amfi_code": fund,

        "average_daily_return": mean_return,

        "risk_std_dev": std_return,

        "sharpe_ratio": sharpe

    })

# ---------------------------------
# Create DataFrame
# ---------------------------------

sharpe_df = pd.DataFrame(results)

# ---------------------------------
# Rank Funds
# ---------------------------------

sharpe_df = sharpe_df.sort_values(
    by="sharpe_ratio",
    ascending=False
)

sharpe_df["rank"] = range(
    1,
    len(sharpe_df) + 1
)

# ---------------------------------
# Display Results
# ---------------------------------

print("\nSharpe Ratio Ranking\n")

print(sharpe_df)

# ---------------------------------
# Save CSV
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\sharpe_ratio.csv"

sharpe_df.to_csv(
    output_file,
    index=False
)

print("\nSharpe Ratio file saved successfully!")

print(output_file)
plt.figure(figsize=(12,8))

plt.barh(
    sharpe_df["amfi_code"].astype(str),
    sharpe_df["sharpe_ratio"]
)

plt.title(
    "Sharpe Ratio Ranking of Mutual Funds",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Sharpe Ratio")

plt.ylabel("AMFI Code")

plt.tight_layout()

plt.show()


## Task 4: Sortino Ratio

Compute downside-risk-adjusted return.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------
# Load Daily Returns Dataset
# ---------------------------------

file_path = r"D:\Mutual_Fund_Analytics\data\processed\daily_returns.csv"

df = pd.read_csv(file_path)

# ---------------------------------
# Risk-Free Rate
# ---------------------------------

risk_free_rate = 0.065

daily_rf = risk_free_rate / 252

# ---------------------------------
# Calculate Sortino Ratio
# ---------------------------------

results = []

for fund in df["amfi_code"].unique():

    fund_data = df[df["amfi_code"] == fund]

    mean_return = fund_data["daily_return"].mean()

    downside_returns = fund_data[
        fund_data["daily_return"] < 0
    ]["daily_return"]

    downside_std = downside_returns.std()

    if downside_std == 0 or np.isnan(downside_std):
        sortino = np.nan
    else:
        sortino = (
            (mean_return - daily_rf)
            / downside_std
        ) * np.sqrt(252)

    results.append({

        "amfi_code": fund,

        "average_daily_return": mean_return,

        "downside_std_dev": downside_std,

        "sortino_ratio": sortino

    })

# ---------------------------------
# Create DataFrame
# ---------------------------------

sortino_df = pd.DataFrame(results)

# ---------------------------------
# Rank Funds
# ---------------------------------

sortino_df = sortino_df.sort_values(
    by="sortino_ratio",
    ascending=False
)

sortino_df["rank"] = range(
    1,
    len(sortino_df) + 1
)

# ---------------------------------
# Display Results
# ---------------------------------

print("\nSortino Ratio Ranking\n")

print(sortino_df)

# ---------------------------------
# Save CSV
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\sortino_ratio.csv"

sortino_df.to_csv(
    output_file,
    index=False
)

print("\nSortino Ratio file saved successfully!")

print(output_file)

# ---------------------------------
# Plot Ranking
# ---------------------------------

plt.figure(figsize=(12,8))

plt.barh(
    sortino_df["amfi_code"].astype(str),
    sortino_df["sortino_ratio"]
)

plt.title(
    "Sortino Ratio Ranking of Mutual Funds",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Sortino Ratio")

plt.ylabel("AMFI Code")

plt.tight_layout()

plt.show()


## Task 5: Alpha & Beta

Perform OLS regression against NIFTY100 benchmark.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ---------------------------------
# Load Daily Returns
# ---------------------------------

fund_df = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\daily_returns.csv"
)

benchmark_df = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\10_benchmark_indices_clean.csv"
)

# ---------------------------------
# Convert Date Columns
# ---------------------------------

fund_df["date"] = pd.to_datetime(fund_df["date"])

benchmark_df["date"] = pd.to_datetime(
    benchmark_df["date"]
)

# ---------------------------------
# Filter Nifty 100
# ---------------------------------

benchmark_df = benchmark_df[
    benchmark_df["index_name"] == "NIFTY100"
]

# ---------------------------------
# Compute Benchmark Daily Returns
# ---------------------------------

benchmark_df = benchmark_df.sort_values("date")

benchmark_df["benchmark_return"] = (
    benchmark_df["close_value"]
    .pct_change()
)

benchmark_df = benchmark_df.dropna()

# ---------------------------------
# Alpha Beta Calculation
# ---------------------------------

results = []

for fund in fund_df["amfi_code"].unique():

    temp = fund_df[
        fund_df["amfi_code"] == fund
    ]

    merged = pd.merge(
        temp,
        benchmark_df[
            ["date", "benchmark_return"]
        ],
        on="date",
        how="inner"
    )

    if len(merged) < 30:
        continue

    slope, intercept, r_value, p_value, std_err = linregress(

        merged["benchmark_return"],

        merged["daily_return"]

    )

    alpha = intercept * 252

    beta = slope

    results.append({

        "amfi_code": fund,

        "alpha": alpha,

        "beta": beta,

        "r_squared": r_value ** 2

    })

# ---------------------------------
# Create DataFrame
# ---------------------------------

alpha_beta_df = pd.DataFrame(results)

alpha_beta_df = alpha_beta_df.sort_values(
    by="alpha",
    ascending=False
)

# ---------------------------------
# Display Results
# ---------------------------------

print("\nAlpha Beta Table\n")

print(alpha_beta_df)

# ---------------------------------
# Save CSV
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\alpha_beta.csv"

alpha_beta_df.to_csv(
    output_file,
    index=False
)

print("\nAlpha Beta file saved successfully!")

print(output_file)

# ---------------------------------
# Plot Alpha
# ---------------------------------

plt.figure(figsize=(12,8))

plt.barh(
    alpha_beta_df["amfi_code"].astype(str),

    alpha_beta_df["alpha"]
)

plt.title(
    "Alpha of Mutual Funds",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Alpha")

plt.ylabel("AMFI Code")

plt.tight_layout()

plt.show()


## Task 6: Maximum Drawdown

Measure the largest decline from a running peak.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------
# Load Cleaned NAV Dataset
# ---------------------------------

file_path = r"D:\Mutual_Fund_Analytics\data\processed\02_nav_history_clean.csv"

df = pd.read_csv(file_path)

# ---------------------------------
# Convert Date Column
# ---------------------------------

df["date"] = pd.to_datetime(df["date"])

# ---------------------------------
# Sort Data
# ---------------------------------

df = df.sort_values(
    by=["amfi_code", "date"]
)

# ---------------------------------
# Calculate Maximum Drawdown
# ---------------------------------

results = []

for fund in df["amfi_code"].unique():

    fund_data = df[
        df["amfi_code"] == fund
    ].copy()

    # Running Maximum NAV

    fund_data["running_max"] = (
        fund_data["nav"].cummax()
    )

    # Drawdown

    fund_data["drawdown"] = (
        fund_data["nav"]
        /
        fund_data["running_max"]
    ) - 1

    # Maximum Drawdown

    max_dd = fund_data["drawdown"].min()

    # Worst Drawdown Date

    worst_date = fund_data.loc[
        fund_data["drawdown"].idxmin(),
        "date"
    ]

    results.append({

        "amfi_code": fund,

        "maximum_drawdown": max_dd,

        "worst_drawdown_date": worst_date

    })

# ---------------------------------
# Create DataFrame
# ---------------------------------

drawdown_df = pd.DataFrame(results)

drawdown_df = drawdown_df.sort_values(
    by="maximum_drawdown"
)

# ---------------------------------
# Display Results
# ---------------------------------

print("\nMaximum Drawdown Table\n")

print(drawdown_df)

# ---------------------------------
# Save CSV
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\maximum_drawdown.csv"

drawdown_df.to_csv(
    output_file,
    index=False
)

print("\nMaximum Drawdown file saved successfully!")

print(output_file)

# ---------------------------------
# Plot Maximum Drawdown
# ---------------------------------

plt.figure(figsize=(12,8))

plt.barh(
    drawdown_df["amfi_code"].astype(str),
    drawdown_df["maximum_drawdown"] * 100
)

plt.title(
    "Maximum Drawdown of Mutual Funds",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Maximum Drawdown (%)")

plt.ylabel("AMFI Code")

plt.tight_layout()

plt.show()


## Task 7: Fund Scorecard

Combine weighted ranks into a 0–100 score.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------
# Load Datasets
# ---------------------------------

cagr = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\cagr_comparison.csv"
)

sharpe = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\sharpe_ratio.csv"
)

alpha = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\alpha_beta.csv"
)

drawdown = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\maximum_drawdown.csv"
)

fund = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\01_fund_master_clean.csv"
)

# ---------------------------------
# Select Required Columns
# ---------------------------------

scorecard = cagr[
    ["amfi_code", "cagr_3yr"]
]

scorecard = scorecard.merge(
    sharpe[
        ["amfi_code", "sharpe_ratio"]
    ],
    on="amfi_code"
)

scorecard = scorecard.merge(
    alpha[
        ["amfi_code", "alpha"]
    ],
    on="amfi_code"
)

scorecard = scorecard.merge(
    drawdown[
        ["amfi_code", "maximum_drawdown"]
    ],
    on="amfi_code"
)

scorecard = scorecard.merge(
    fund[
        ["amfi_code", "expense_ratio_pct"]
    ],
    on="amfi_code"
)

# ---------------------------------
# Ranking
# ---------------------------------

scorecard["rank_cagr"] = (
    scorecard["cagr_3yr"]
    .rank(
        ascending=False,
        method="min"
    )
)

scorecard["rank_sharpe"] = (
    scorecard["sharpe_ratio"]
    .rank(
        ascending=False,
        method="min"
    )
)

scorecard["rank_alpha"] = (
    scorecard["alpha"]
    .rank(
        ascending=False,
        method="min"
    )
)

scorecard["rank_expense"] = (
    scorecard["expense_ratio_pct"]
    .rank(
        ascending=True,
        method="min"
    )
)

scorecard["rank_drawdown"] = (
    scorecard["maximum_drawdown"]
    .rank(
        ascending=False,
        method="min"
    )
)

# ---------------------------------
# Composite Score
# ---------------------------------

scorecard["fund_score"] = (

      (41 - scorecard["rank_cagr"]) * 0.30

    + (41 - scorecard["rank_sharpe"]) * 0.25

    + (41 - scorecard["rank_alpha"]) * 0.20

    + (41 - scorecard["rank_expense"]) * 0.15

    + (41 - scorecard["rank_drawdown"]) * 0.10

)

# ---------------------------------
# Convert Score to 100 Scale
# ---------------------------------

scorecard["fund_score"] = (

    scorecard["fund_score"]

    / scorecard["fund_score"].max()

) * 100

# ---------------------------------
# Final Ranking
# ---------------------------------

scorecard = scorecard.sort_values(

    by="fund_score",

    ascending=False

)

scorecard["overall_rank"] = range(

    1,

    len(scorecard) + 1

)

# ---------------------------------
# Display Results
# ---------------------------------

print("\nFund Scorecard\n")

print(scorecard)

# ---------------------------------
# Save CSV
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\fund_scorecard.csv"

scorecard.to_csv(
    output_file,
    index=False
)

print("\nFund Scorecard saved successfully!")

print(output_file)

# ---------------------------------
# Plot Top 10 Funds
# ---------------------------------

top10 = scorecard.head(10)

plt.figure(figsize=(12,6))

plt.bar(

    top10["amfi_code"].astype(str),

    top10["fund_score"]

)

plt.title(

    "Top 10 Mutual Funds by Composite Score",

    fontsize=16,

    fontweight="bold"

)

plt.xlabel("AMFI Code")

plt.ylabel("Fund Score")

plt.tight_layout()

plt.show()


## Task 8: Benchmark Comparison & Tracking Error

Compare Top 5 funds with NIFTY50/NIFTY100 and compute tracking error.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------
# Load Datasets
# ---------------------------------

scorecard = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\fund_scorecard.csv"
)

fund_nav = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\02_nav_history_clean.csv"
)

daily_returns = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\daily_returns.csv"
)

benchmark = pd.read_csv(
    r"D:\Mutual_Fund_Analytics\data\processed\10_benchmark_indices_clean.csv"
)

# ---------------------------------
# Convert Dates
# ---------------------------------

fund_nav["date"] = pd.to_datetime(fund_nav["date"])

daily_returns["date"] = pd.to_datetime(daily_returns["date"])

benchmark["date"] = pd.to_datetime(benchmark["date"])

# ---------------------------------
# Select Top 5 Funds
# ---------------------------------

top5 = scorecard.head(5)["amfi_code"].tolist()

# ---------------------------------
# Plot NAV Comparison
# ---------------------------------

plt.figure(figsize=(14,7))

for fund in top5:

    temp = fund_nav[
        fund_nav["amfi_code"] == fund
    ]

    plt.plot(
        temp["date"],
        temp["nav"],
        label=str(fund)
    )

# Plot Nifty 50

nifty50 = benchmark[
    benchmark["index_name"] == "NIFTY50"
]

plt.plot(
    nifty50["date"],
    nifty50["close_value"],
    linestyle="--",
    linewidth=2,
    label="NIFTY50"
)

# Plot Nifty 100

nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY100"
]

plt.plot(
    nifty100["date"],
    nifty100["close_value"],
    linestyle=":",
    linewidth=2,
    label="NIFTY100"
)

plt.title(
    "Top 5 Funds vs Nifty 50 & Nifty 100",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Date")

plt.ylabel("NAV / Index Value")

plt.legend()

plt.tight_layout()

plt.show()

# ---------------------------------
# Benchmark Daily Returns
# ---------------------------------

tracking = []

for benchmark_name in ["NIFTY50", "NIFTY100"]:

    bench = benchmark[
        benchmark["index_name"] == benchmark_name
    ].copy()

    bench = bench.sort_values("date")

    bench["benchmark_return"] = (
        bench["close_value"].pct_change()
    )

    bench = bench.dropna()

    for fund in top5:

        fund_data = daily_returns[
            daily_returns["amfi_code"] == fund
        ]

        merged = pd.merge(
            fund_data,
            bench[
                ["date", "benchmark_return"]
            ],
            on="date",
            how="inner"
        )

        tracking_error = (
            np.std(
                merged["daily_return"]
                -
                merged["benchmark_return"]
            )
            * np.sqrt(252)
        )

        tracking.append({

            "amfi_code": fund,

            "benchmark": benchmark_name,

            "tracking_error": tracking_error

        })

# ---------------------------------
# Create Tracking Error Table
# ---------------------------------

tracking_df = pd.DataFrame(tracking)

print("\nTracking Error Table\n")

print(tracking_df)

# ---------------------------------
# Save CSV
# ---------------------------------

output_file = r"D:\Mutual_Fund_Analytics\data\processed\tracking_error.csv"

tracking_df.to_csv(
    output_file,
    index=False
)

print("\nTracking Error file saved successfully!")

print(output_file)


# Final Summary

## Project Overview

This notebook presents a comprehensive performance analytics study of 40 mutual fund schemes using Python. The analysis was performed on cleaned mutual fund datasets covering NAV history, benchmark indices, fund information, and performance metrics. The objective was to evaluate mutual fund performance using return-based, risk-based, and benchmark comparison techniques.

---

## Work Completed

The following analyses were successfully completed:

### 1. Daily Return Analysis
- Calculated daily percentage returns for all 40 mutual fund schemes.
- Validated the return distribution using a histogram.

### 2. CAGR Analysis
- Computed the Compound Annual Growth Rate (CAGR) for 1-year, 3-year, and 5-year periods.
- Compared long-term annualized performance across all funds.

### 3. Sharpe Ratio
- Calculated the Sharpe Ratio using a 6.5% annual risk-free rate.
- Ranked all funds based on risk-adjusted returns.

### 4. Sortino Ratio
- Calculated the Sortino Ratio using downside standard deviation.
- Evaluated downside risk-adjusted performance.

### 5. Alpha and Beta
- Performed Ordinary Least Squares (OLS) regression against the NIFTY100 benchmark.
- Computed Alpha, Beta, and R² values for all funds.

### 6. Maximum Drawdown
- Calculated the largest decline from the historical peak NAV for every fund.
- Identified the worst drawdown period.

### 7. Fund Scorecard
- Developed a composite score (0–100) using weighted rankings of:
  - 3-Year CAGR
  - Sharpe Ratio
  - Alpha
  - Expense Ratio
  - Maximum Drawdown
- Ranked all mutual funds based on overall performance.

### 8. Benchmark Comparison
- Compared the Top 5 mutual funds with NIFTY50 and NIFTY100.
- Calculated Tracking Error to measure deviation from benchmark performance.

---

## Key Outcomes

- Successfully analyzed all 40 mutual fund schemes.
- Evaluated return, volatility, downside risk, benchmark performance, and historical losses.
- Developed a comprehensive ranking methodology using multiple financial metrics.
- Generated visualizations and analytical reports for investment performance evaluation.

---

## Deliverables Generated

- daily_returns.csv
- cagr_comparison.csv
- sharpe_ratio.csv
- sortino_ratio.csv
- alpha_beta.csv
- maximum_drawdown.csv
- fund_scorecard.csv
- tracking_error.csv
- Benchmark Comparison Chart
- Performance_Analytics.ipynb

---

## Technologies Used

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- SciPy
- Jupyter Notebook

---

## Conclusion

This Performance Analytics module demonstrates a complete end-to-end financial analysis workflow for mutual funds. By combining return analysis, risk assessment, benchmark comparison, and composite scoring, the project provides meaningful insights into fund performance and supports informed investment decision-making. The notebook serves as a comprehensive analytical report and showcases practical implementation of financial performance metrics using Python.

# Conclusion

This notebook documents the complete Day 4 Performance Analytics workflow.
